In [1]:
import torch
import torch.nn as nn
import numpy as np 
import matplotlib.pyplot as plt
import sys 
import os
import glob

In [2]:
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/transforms")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/configs")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/plots")

In [3]:
from seg_recon_vit3d_overlap import *
from utils.read_yaml import read_yaml
from utils.model_select import model_select

In [4]:
base_path = "/media/ana-caznok/SSD-08/recon-segment/"
config = read_yaml(base_path + 'configs/restormer-seg_msi2mask4.yaml')
model,load = model_select(config)

Looking for checkpoint in models/restormer-seg_msi2mask4.pth. Exact path only: True.
No checkpoint found or error loading — starting from scratch.
Could't find checkpoint models/restormer-seg_msi2mask4.pth! :(
Selected model restormer-seg_4to61_complex: Restormer_Seg


In [5]:
def get_output_shape(transform_class, transform_params, input_shape):
    """
    Computes the output shape of a PyTorch nn transform given the transform class, 
    its parameters, and the input tensor shape.

    Parameters:
    - transform_class (torch.nn.Module): The class of the PyTorch transform (e.g., nn.Conv2d).
    - transform_params (dict): Parameters required to instantiate the transform.
    - input_shape (tuple): The shape of the input tensor (e.g., (1, 3, 224, 224)).

    Returns:
    - tuple: Output tensor shape after applying the transform.
    """

    # Instantiate the transform using the provided class and parameters
    transform = transform_class(**transform_params)

    # Create a dummy input tensor with the specified input shape
    dummy_input = torch.randn(*input_shape)

    # Apply the transform to the dummy input without tracking gradients
    with torch.no_grad():
        output = transform(dummy_input)

    # Return the output shape as a tuple
    return tuple(output.shape)


In [6]:
model.encoder

Encoder(
  (patch_embed): OverlapPatchEmbed(
    (proj): Conv2d(4, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (encoder_level1): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm(
        (body): WithBias_LayerNorm()
      )
      (attn): Attention(
        (qkv): Conv2d(48, 144, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (qkv_dwconv): Conv2d(144, 144, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=144, bias=False)
        (project_out): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
      (norm2): LayerNorm(
        (body): WithBias_LayerNorm()
      )
      (ffn): FeedForward(
        (project_in): Conv2d(48, 254, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (dwconv): Conv2d(254, 254, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=254, bias=False)
        (project_out): Conv2d(127, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
    )
    (1): TransformerBlock(
   

In [7]:
B, C, Y, X = 1, 4, 256, 256
E = 768
img = torch.randn(*(B,C,Y,X))

In [8]:
out_enc_level1, out_enc_level2, out_enc_level3, latent = model.encoder(img)

In [11]:
model.seg_decoder(latent).shape

torch.Size([1, 1, 256, 256])

In [ ]:
dim=48
int(dim * 2 ** 1)

In [ ]:
dim*8

In [ ]:
384*32*32 /(256*256*1)

In [ ]:
out_enc_level1.shape

In [ ]:
x,h,w = model.encoder(img)

In [ ]:
b, t, e = x.shape
decoder_input = x.transpose(1, 2).contiguous().view(B, E, h, w)
b_d, e_d, h, w = decoder_input.shape
decoder_input_shape = (b_d,e_d,h, w)

In [ ]:
decoder_input_shape

In [ ]:
x.transpose(1, 2).contiguous().view(B, E, h, w).shape

In [ ]:
for i in range(len(channels) - 1):
            layers.append(nn.ConvTranspose2d(
                in_channels=channels[i],
                out_channels=channels[i+1],
                kernel_size=4,
                stride=2,
                padding=1
            ))

In [ ]:
upsample_factor = model.decoder.upsample_factor
upsample_layers = model.decoder.num_upsample_layers 
channels =  model.decoder.dec_channels

In [ ]:
decoder_input_shape = (b_d,e_d,h, w)
conv_shapes = [decoder_input_shape]
for i in range(len(channels)-1): 
  conv_output_shape = get_output_shape(
          nn.ConvTranspose2d,
          {"in_channels": channels[i],
            "out_channels": channels[i+1], 
            "kernel_size": 5,
            "stride": 2,
            "padding": 1},
          decoder_input_shape
      )
  conv_shapes.append(conv_output_shape)
  decoder_input_shape = conv_output_shape
#print("Output shape:", conv_output_shape)

In [ ]:
conv_output_shape = get_output_shape(
                    nn.Conv2d,
                    {"in_channels": channels[-1],
                        "out_channels": 61, 
                        "kernel_size": 2,
                        "padding": 1},
                    decoder_input_shape) 
conv_shapes.append(conv_output_shape)

In [ ]:
#kernel size: ou 5 ou 6

In [ ]:
conv_shapes

In [ ]:
256 + 256/2